# Lab 5 - Compare: accuracy, tokens and latency by problem type

Every model scored on the **same 1,184 benchmark cases** (`data/reference_results.csv`, from the
instructor's final evaluation):

| Model | Trained on |
|---|---|
| Base | - |
| Attendee-scale SFT / SFT -> RLVR (1K) | 1,000 raw requests: your data size, same pipeline |
| Instructor SFT / SFT -> RLVR (20K) | 20,000 raw requests |

| Track | What it tests |
|---|---|
| Typical | the natural request mix |
| Hard, 40 / 80 candidates | hard requests, with known and never-seen products, and with twice the traps |
| Extreme | the hardest scenarios (difficulty 70+) |

**Accuracy** = every hard rule passes (checked in code). **Latency** is estimated from tokens at the
endpoint's measured ~24 tokens/s. Look at the problem types separately: the overall average hides
where RL helps. If you turned on `EVALUATE` in Lab 3, your own models join the tables when their
evaluation finishes.

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath("../src"))
%load_ext autoreload
%autoreload 2
import pandas as pd
pd.set_option("display.max_colwidth", 120)
from octank import config, state
S = config.get_settings("attendee")
ST = state.load(S)
print(f"region={S.region}  bucket={S.bucket}  prefix={S.prefix}  run={S.run_name}")
print(f"role={S.role_arn}")

In [ ]:
from octank import arms, benchmarks, workflow
jobs = arms.status(S)
print("Your jobs:"); display(jobs if len(jobs) else "none launched (Lab 3)")
ref = pd.read_csv(config.DATA_DIR / "reference_results.csv")
mine = arms.results(S)                                   # only if you evaluated your models
df = pd.concat([ref, mine.assign(model="my_" + mine["model"]) if len(mine) else mine], ignore_index=True)
ORDER = [m for m in ["base", "attendee_scale_sft", "attendee_scale_sft_rlvr", "my_sft", "my_sft_rlvr",
                     "instructor_sft", "instructor_sft_rlvr"] if m in set(df.model)]
print("models:", ORDER, "| cases per model:", df.groupby("model").size().to_dict())

In [ ]:
t = workflow.by_problem_type(df)
TPS = 24.0
t["latency_s"] = t["tokens"] / TPS
pivot = lambda col: t.pivot(index="problem_type", columns="model", values=col).reindex(
    [p for p, _ in benchmarks.PROBLEM_TYPES])[ORDER]
print("Accuracy (rule pass)"); display(pivot("rule_pass").style.format("{:.1%}"))
print("Average tokens");       display(pivot("tokens").round(0))
print("Estimated latency (s)"); display(pivot("latency_s").round(0))
print("Cut off (no answer)");  display(pivot("cut_off").style.format("{:.1%}"))

## 1. Where does each stage help?

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
pivot("rule_pass").plot.bar(ax=axes[0], rot=15, width=0.85)
axes[0].set_ylim(0, 1); axes[0].set_ylabel("rule pass"); axes[0].set_title("Accuracy by problem type")
pivot("tokens").plot.bar(ax=axes[1], rot=15, width=0.85)
axes[1].set_ylabel("tokens"); axes[1].set_title("Average tokens by problem type")
for ax in axes: ax.legend(fontsize=8)
plt.tight_layout()

## 2. SFT vs SFT -> RLVR on the same cases

Paired comparison: *fixed* = cases the RLVR model passes and SFT fails; *broke* = the reverse.
p is an exact sign test - with only a few hundred cases, small differences are noise. Compare the
two scales: with 20x the data RL has a strong SFT model to sharpen; at 1K data (one RL step) the
gain is smaller and mostly within noise.

In [ ]:
from math import comb
def paired(a, b, tracks):
    p = df[df.track.isin(tracks) & df.model.isin([a, b])].pivot_table(index=["track", "case"], columns="model", values="hard_pass").dropna()
    if p.empty or a not in p or b not in p:
        return None
    x, y = p[a].astype(bool), p[b].astype(bool); up, dn = int((~x & y).sum()), int((x & ~y).sum())
    pv = min(1.0, 2 * sum(comb(up + dn, k) for k in range(min(up, dn) + 1)) / 2 ** (up + dn)) if up + dn else 1.0
    return {"n": len(p), "SFT": f"{x.mean():.1%}", "SFT -> RLVR": f"{y.mean():.1%}", "fixed": up, "broke": dn, "p": round(pv, 4)}
rows = []
for who, a, b in (("instructor (20K)", "instructor_sft", "instructor_sft_rlvr"),
                  ("attendee scale (1K)", "attendee_scale_sft", "attendee_scale_sft_rlvr"),
                  ("yours", "my_sft", "my_sft_rlvr")):
    for name, tracks in benchmarks.PROBLEM_TYPES:
        r = paired(a, b, tracks)
        if r:
            rows.append({"models": who, "problem type": name, **r})
pd.DataFrame(rows)

## 3. Same request, side by side

In [ ]:
from octank import checks as ck
cases = {c["request"]["request_id"]: c for c in benchmarks.cases_40()}
case = df[df.track == "hard_known"]["case"].sample(1, random_state=3).iloc[0]   # change random_state to browse
c = cases[case]
print("SHOPPER:", c["request"]["message"])
print("IMPLIED BY THE SITUATION:", c["request"].get("hidden_avoid") or "-", "\n")
by_id = {x["id"]: x for x in c["candidates"]}
for m in ORDER:
    r = df[(df.model == m) & (df.case == case) & (df.track == "hard_known")]
    if r.empty:
        continue
    r = r.iloc[0]
    print(f"{m:24s} {'PASS' if r.hard_pass else 'FAIL'} {'(cut off)' if r.cut_off else '':10s} {int(r.tokens):5d} tok")